# Instagram Reel Metrics — Google Colab

Fetch **real** Instagram Reel statistics (views, likes, comments, shares, saves, reposts) using Python only — no web browser UI required.

## How to use

1. **Run cells in order** (Runtime → Run all, or Shift+Enter through each cell).
2. **Cell 1** installs dependencies.
3. **Cell 2** creates the `reel_metrics` package automatically (self-contained — no extra uploads needed).
4. **Cell 3** — enter your Instagram login.
5. Pick a mode below: **Single Reel**, **Profile Reels**, or **Bulk CSV**.

> **Note:** Use a real Instagram account. Sessions are cached in `.ig_sessions/` for the current Colab runtime.

See `COLAB_GUIDE.md` in the repo for the full written guide.

In [ ]:
# Step 1 — Install dependencies
!pip install -q instagrapi==2.16.25 pandas matplotlib

import IPython
print('Dependencies installed. Python:', IPython.sys_info()['python_version'])

In [ ]:
"""Bootstrap reel_metrics package (runs automatically if not already present)."""
import os
import sys
import importlib

MODULES = {'reel_metrics/errors.py': '"""Map instagrapi exceptions to human-readable messages."""\n\nfrom instagrapi.exceptions import (\n    BadPassword,\n    ChallengeRequired,\n    ClientError,\n    LoginRequired,\n    PleaseWaitFewMinutes,\n    PrivateAccount,\n    TwoFactorRequired,\n    UserNotFound,\n)\n\n\ndef error_message_from_exception(e: Exception, what: str = "Reel") -> str:\n    """Human-readable error string for scraping failures."""\n    if isinstance(e, UserNotFound):\n        return f"{what} not found."\n    if isinstance(e, PrivateAccount):\n        return f"{what} is private and the logged-in account does not follow it."\n    if isinstance(e, BadPassword):\n        return "Bad password."\n    if isinstance(e, TwoFactorRequired):\n        return "Two-factor authentication required."\n    if isinstance(e, ChallengeRequired):\n        return "Instagram security challenge required — confirm in the app, then retry."\n    if isinstance(e, LoginRequired):\n        return "Login required (session expired). Provide your password."\n    if isinstance(e, PleaseWaitFewMinutes):\n        return "Instagram rate-limited this account. Wait a few minutes."\n    if isinstance(e, ClientError):\n        return f"Instagram refused the request: {e}"\n    return f"{type(e).__name__}: {e}"\n', 'reel_metrics/parsers.py': '"""URL, shortcode, and username parsing."""\n\nimport re\nfrom typing import Optional\n\nfrom instagrapi import Client\n\n_RESERVED_IG_PATHS = {\n    "reel", "reels", "p", "tv", "stories", "explore", "accounts",\n    "direct", "about", "developer", "legal", "press", "api",\n}\n\n_SHORTCODE_RE = re.compile(r"instagram\\.com/(?:reel|reels|p|tv)/([A-Za-z0-9_-]+)")\n_PROFILE_PATH_RE = re.compile(r"instagram\\.com/([A-Za-z0-9_.]+)")\n_IG_MEDIA_URL_RE = re.compile(\n    r"https?://(?:www\\.)?instagram\\.com/(?:reel|reels|p|tv)/[A-Za-z0-9_-]+",\n    re.I,\n)\n\n_URL_COLUMN_NAMES = frozenset({\n    "url", "link", "links", "reel", "reels", "reel_url", "reel_link",\n    "instagram_url", "instagram", "ig_url", "post_url", "media_url",\n})\n\n\ndef extract_shortcode(value: str) -> str:\n    """Accept a raw shortcode or full reel/post URL; return the shortcode."""\n    value = value.strip()\n    match = _SHORTCODE_RE.search(value)\n    return match.group(1) if match else value\n\n\ndef extract_username(value: str) -> Optional[str]:\n    """Accept a username or profile URL; return the username."""\n    value = value.strip().lstrip("@")\n\n    if _SHORTCODE_RE.search(value):\n        return None\n\n    match = _PROFILE_PATH_RE.search(value)\n    if match:\n        candidate = match.group(1)\n        if candidate.lower() in _RESERVED_IG_PATHS:\n            return None\n        return candidate\n\n    return value.rstrip("/")\n\n\ndef resolve_target_username(cl: Client, value: str) -> str:\n    """Return a username even when the user pasted a reel URL."""\n    direct = extract_username(value)\n    if direct:\n        return direct\n\n    shortcode = extract_shortcode(value)\n    media_pk = cl.media_pk_from_code(shortcode)\n    return cl.media_info(media_pk).user.username\n\n\ndef is_valid_media_input(value: str) -> bool:\n    """True when value is an Instagram reel/post URL or a bare shortcode."""\n    value = value.strip()\n    if _IG_MEDIA_URL_RE.search(value) or _SHORTCODE_RE.search(value):\n        return True\n    if re.fullmatch(r"[A-Za-z0-9_-]+", value):\n        return not value.isdigit() and len(value) >= 6\n    return False\n\n\ndef row_is_header(row: list[str]) -> bool:\n    """Detect a header row (column names, not reel data)."""\n    if any(_IG_MEDIA_URL_RE.search(c) or _SHORTCODE_RE.search(c) for c in row):\n        return False\n    return any(c.strip().lower() in _URL_COLUMN_NAMES for c in row)\n\n\ndef find_url_in_row(row: list[str]) -> Optional[str]:\n    """Pick the Instagram reel/post URL from a CSV row."""\n    for cell in row:\n        val = cell.strip()\n        if val and (_IG_MEDIA_URL_RE.search(val) or _SHORTCODE_RE.search(val)):\n            return val\n    non_empty = [c.strip() for c in row if c.strip()]\n    if len(non_empty) == 1 and is_valid_media_input(non_empty[0]):\n        return non_empty[0]\n    return None\n', 'reel_metrics/session.py': '"""Instagram session management with disk persistence."""\n\nimport os\nimport threading\nfrom typing import Optional\n\nfrom instagrapi import Client\n\n_clients: dict[str, Client] = {}\n_clients_lock = threading.Lock()\n\n\ndef default_session_dir() -> str:\n    """Directory for cached instagrapi session files."""\n    base = os.environ.get("IG_SESSION_DIR")\n    if base:\n        return base\n    return os.path.join(os.getcwd(), ".ig_sessions")\n\n\ndef session_path(username: str, session_dir: Optional[str] = None) -> str:\n    """Path to the saved session file for ``username``."""\n    directory = session_dir or default_session_dir()\n    os.makedirs(directory, exist_ok=True)\n    return os.path.join(directory, f"instagrapi-{username}.json")\n\n\ndef new_client() -> Client:\n    """Create a fresh instagrapi Client with sensible defaults."""\n    cl = Client()\n    cl.delay_range = [1, 3]\n    return cl\n\n\ndef get_client(\n    username: str,\n    password: Optional[str] = None,\n    session_dir: Optional[str] = None,\n) -> Client:\n    """\n    Return a logged-in Client for ``username``.\n\n    Reuses in-memory cache, then disk session, then password login.\n    """\n    with _clients_lock:\n        cached = _clients.get(username)\n        if cached is not None:\n            return cached\n\n        sess_file = session_path(username, session_dir)\n        cl = new_client()\n\n        if os.path.exists(sess_file):\n            try:\n                cl.load_settings(sess_file)\n                cl.get_timeline_feed()\n                _clients[username] = cl\n                return cl\n            except Exception:\n                cl = new_client()\n                try:\n                    cl.load_settings(sess_file)\n                except Exception:\n                    pass\n\n        if not password:\n            raise RuntimeError(\n                "No valid cached session for this username. Provide a password to log in."\n            )\n\n        cl.login(username, password)\n        try:\n            cl.dump_settings(sess_file)\n        except Exception:\n            pass\n\n        _clients[username] = cl\n        return cl\n', 'reel_metrics/scraping.py': '"""Media metrics, comments, and profile reel fetching."""\n\nfrom datetime import datetime, timezone\nfrom typing import Optional\n\nfrom instagrapi import Client\nfrom instagrapi.exceptions import ClientError, CommentUnavailable, CommentsDisabled\nfrom instagrapi.extractors import extract_comment\n\n\ndef media_to_dict(media) -> dict:\n    """Light-weight extraction from an instagrapi Media object."""\n    taken_at = getattr(media, "taken_at", None)\n    views = getattr(media, "play_count", None) or getattr(media, "view_count", None)\n\n    return {\n        "shortcode": getattr(media, "code", None),\n        "owner": media.user.username if getattr(media, "user", None) else None,\n        "views": int(views) if views else None,\n        "likes": int(getattr(media, "like_count", 0) or 0),\n        "comments": int(getattr(media, "comment_count", 0) or 0),\n        "date": taken_at.strftime("%Y-%m-%d %H:%M:%S") if taken_at else None,\n        "is_video": getattr(media, "media_type", 0) == 2,\n        "caption": getattr(media, "caption_text", "") or "",\n    }\n\n\ndef _as_int(value) -> Optional[int]:\n    if value is None:\n        return None\n    try:\n        return int(value)\n    except (TypeError, ValueError):\n        return None\n\n\ndef _extract_repost_count(item: dict) -> Optional[int]:\n    for key in (\n        "media_repost_count",\n        "repost_count",\n        "organic_media_repost_count",\n        "reposts_count",\n    ):\n        if key in item:\n            val = _as_int(item.get(key))\n            if val is not None:\n                return val\n    for key, value in item.items():\n        if isinstance(value, (dict, list)):\n            continue\n        kl = key.lower()\n        if "repost" in kl and "count" in kl:\n            val = _as_int(value)\n            if val is not None:\n                return val\n    return None\n\n\ndef media_metric_fields(m: dict) -> dict:\n    """Shared metric subset for single-reel and bulk responses."""\n    return {\n        "views": m.get("views"),\n        "likes": m.get("likes"),\n        "comments": m.get("comments"),\n        "shares": m.get("shares"),\n        "saves": m.get("saves"),\n        "reposts": m.get("reposts"),\n        "date": m.get("date"),\n    }\n\n\ndef fetch_single_media(cl: Client, media_pk) -> dict:\n    """Full extraction via the mobile API (shares, saves, reposts included)."""\n    raw = cl.private_request(f"media/{media_pk}/info/")\n    if not isinstance(raw, dict) or not raw.get("items"):\n        raise RuntimeError("Media not found in response")\n\n    item = raw["items"][0]\n    user_obj = item.get("user") or {}\n    caption_obj = item.get("caption") or {}\n    caption_text = caption_obj.get("text", "") if isinstance(caption_obj, dict) else ""\n\n    date_str = None\n    if (ts := item.get("taken_at")):\n        try:\n            date_str = datetime.fromtimestamp(int(ts)).strftime("%Y-%m-%d %H:%M:%S")\n        except Exception:\n            pass\n\n    views = (\n        item.get("play_count")\n        or item.get("ig_play_count")\n        or item.get("view_count")\n    )\n\n    return {\n        "shortcode": item.get("code"),\n        "owner": user_obj.get("username"),\n        "likes": _as_int(item.get("like_count")),\n        "comments": _as_int(item.get("comment_count")),\n        "views": _as_int(views),\n        "ig_views": _as_int(item.get("ig_play_count")),\n        "shares": _as_int(item.get("reshare_count")),\n        "saves": _as_int(item.get("save_count")),\n        "reposts": _extract_repost_count(item),\n        "date": date_str,\n        "is_video": item.get("media_type") == 2,\n        "caption": caption_text,\n    }\n\n\ndef comment_to_dict(comment, *, is_reply: bool = False, parent_pk=None) -> dict:\n    user = getattr(comment, "user", None)\n    created = getattr(comment, "created_at_utc", None)\n    return {\n        "pk": str(comment.pk),\n        "username": user.username if user else None,\n        "full_name": getattr(user, "full_name", None) if user else None,\n        "text": comment.text or "",\n        "likes": comment.like_count,\n        "date": created.strftime("%Y-%m-%d %H:%M:%S") if created else None,\n        "is_reply": is_reply,\n        "parent_pk": str(parent_pk) if parent_pk else None,\n    }\n\n\ndef fetch_media_comments(cl: Client, media_pk) -> tuple[list[dict], Optional[str]]:\n    """Fetch every comment on a media item (top-level + replies)."""\n    media_id = cl.media_id(str(media_pk))\n    comments: list[dict] = []\n    note: Optional[str] = None\n    min_id = ""\n    max_id = ""\n    seen_pks: set[str] = set()\n\n    def append_replies(parent_pk: str, child_count: int) -> None:\n        if not child_count:\n            return\n        try:\n            replies = cl.media_comment_replies(media_id, parent_pk, amount=0)\n        except Exception:\n            return\n        for reply in replies:\n            comments.append(comment_to_dict(reply, is_reply=True, parent_pk=parent_pk))\n\n    try:\n        for _ in range(500):\n            params: dict = {\n                "can_support_threading": "true",\n                "permalink_enabled": "false",\n            }\n            if min_id:\n                params["min_id"] = min_id\n            if max_id:\n                params["max_id"] = max_id\n\n            result = cl.private_request(f"media/{media_id}/comments/", params)\n            page = result.get("comments") or []\n            if not page:\n                break\n\n            new_on_page = 0\n            for raw in page:\n                pk = str(raw.get("pk", ""))\n                if not pk or pk in seen_pks:\n                    continue\n                seen_pks.add(pk)\n                new_on_page += 1\n                comment = extract_comment(raw)\n                comments.append(comment_to_dict(comment))\n                append_replies(str(comment.pk), int(raw.get("child_comment_count") or 0))\n\n            if new_on_page == 0:\n                break\n\n            min_id = result.get("next_min_id") or result.get("min_id") or ""\n            max_id = result.get("next_max_id") or result.get("max_id") or ""\n\n    except CommentsDisabled:\n        return [], "comments_disabled"\n    except CommentUnavailable:\n        return [], "comments_unavailable"\n    except ClientError:\n        note = "partial" if comments else None\n        if not comments:\n            return [], "partial"\n\n    return comments, note\n\n\ndef reel_dict_without_comments(reel_dict: dict) -> dict:\n    out = dict(reel_dict)\n    out["reel_comments"] = []\n    out["comments_fetched"] = 0\n    out["comments_note"] = None\n    return out\n\n\ndef iter_user_clips_v1(cl: Client, user_id, limit: int = 0, page_size: int = 12):\n    """Yield profile reels page-by-page from Instagram."""\n    next_cursor = ""\n    yielded = 0\n    while True:\n        remaining = None if limit <= 0 else limit - yielded\n        if remaining is not None and remaining <= 0:\n            break\n\n        fetch_amount = page_size if remaining is None else min(page_size, remaining)\n        medias_page, next_cursor = cl.user_clips_paginated_v1(\n            user_id, amount=fetch_amount, end_cursor=next_cursor\n        )\n        if not medias_page:\n            break\n\n        for media in medias_page:\n            yield media\n            yielded += 1\n            if limit > 0 and yielded >= limit:\n                return\n\n        if not next_cursor:\n            break\n\n\ndef media_to_reel_metrics(media) -> dict:\n    try:\n        return reel_dict_without_comments(media_to_dict(media))\n    except Exception as inner:\n        return reel_dict_without_comments({\n            "shortcode": getattr(media, "code", "?"),\n            "date": None,\n            "views": None,\n            "likes": None,\n            "comments": None,\n            "caption": f"[error reading post: {inner}]",\n            "is_video": None,\n        })\n\n\ndef compute_reel_date_range(reels_raw) -> dict:\n    """Compute oldest / newest / span across a list of Media objects."""\n    dates = []\n    for m in reels_raw or []:\n        t = getattr(m, "taken_at", None)\n        if isinstance(t, datetime):\n            if t.tzinfo is None:\n                t = t.replace(tzinfo=timezone.utc)\n            dates.append(t)\n\n    if not dates:\n        return {\n            "oldest_iso": None, "newest_iso": None,\n            "oldest_display": None, "newest_display": None,\n            "oldest_date": None, "newest_date": None,\n            "span_days": 0, "count": 0,\n        }\n\n    oldest = min(dates).astimezone(timezone.utc)\n    newest = max(dates).astimezone(timezone.utc)\n    return {\n        "oldest_iso": oldest.isoformat(),\n        "newest_iso": newest.isoformat(),\n        "oldest_display": oldest.strftime("%b %d, %Y %H:%M UTC"),\n        "newest_display": newest.strftime("%b %d, %Y %H:%M UTC"),\n        "oldest_date": oldest.strftime("%b %d, %Y"),\n        "newest_date": newest.strftime("%b %d, %Y"),\n        "span_days": (newest.date() - oldest.date()).days,\n        "count": len(dates),\n    }\n', 'reel_metrics/csv_io.py': '"""CSV URL extraction for bulk reel fetching."""\n\nimport csv\nimport io\n\nfrom .parsers import _URL_COLUMN_NAMES, find_url_in_row, row_is_header\n\n\ndef parse_urls_from_csv(file_bytes: bytes) -> list[str]:\n    """\n    Extract reel URLs from a CSV file.\n\n    Priority:\n      1. A named URL column (url, link, reel_url, etc.)\n      2. Any cell containing an Instagram reel/post URL\n      3. A single bare shortcode in the row\n    """\n    text = file_bytes.decode("utf-8-sig")\n    reader = csv.reader(io.StringIO(text))\n    rows = [r for r in reader if any(cell.strip() for cell in r)]\n    if not rows:\n        return []\n\n    header = [c.strip().lower() for c in rows[0]]\n    url_col_idx = next(\n        (i for i, h in enumerate(header) if h in _URL_COLUMN_NAMES),\n        None,\n    )\n\n    if url_col_idx is not None and row_is_header(rows[0]):\n        urls: list[str] = []\n        for row in rows[1:]:\n            if url_col_idx < len(row):\n                val = row[url_col_idx].strip()\n                if val:\n                    urls.append(val)\n        if urls:\n            return urls\n\n    data_start = 1 if row_is_header(rows[0]) else 0\n    urls = []\n    for row in rows[data_start:]:\n        found = find_url_in_row(row)\n        if found:\n            urls.append(found)\n    return urls\n\n\ndef parse_urls_from_csv_path(path: str) -> list[str]:\n    """Read a CSV file from disk and extract reel URLs."""\n    with open(path, "rb") as f:\n        return parse_urls_from_csv(f.read())\n', 'reel_metrics/display.py': '"""Terminal and notebook display helpers (tables, summaries, charts)."""\n\nfrom typing import Any, Optional\n\nimport pandas as pd\n\n\ndef _fmt_int(value: Any) -> str:\n    if value is None:\n        return "—"\n    try:\n        return f"{int(value):,}"\n    except (TypeError, ValueError):\n        return str(value)\n\n\ndef print_section(title: str) -> None:\n    print()\n    print("=" * 72)\n    print(title)\n    print("=" * 72)\n\n\ndef show_profile_summary(profile: dict, date_range: Optional[dict] = None) -> None:\n    """Print profile metadata and optional date span."""\n    print_section("Profile Summary")\n    print(f"  Username:   {profile.get(\'username\')}")\n    print(f"  Full name:  {profile.get(\'full_name\')}")\n    print(f"  Followers:  {_fmt_int(profile.get(\'followers\'))}")\n    print(f"  Following:  {_fmt_int(profile.get(\'following\'))}")\n    print(f"  Posts:      {_fmt_int(profile.get(\'posts\'))}")\n    print(f"  Private:    {profile.get(\'is_private\')}")\n    if date_range and date_range.get("count"):\n        print(f"  Reel span:  {date_range.get(\'oldest_date\')} → {date_range.get(\'newest_date\')}")\n        print(f"  Span days:  {date_range.get(\'span_days\')}")\n        print(f"  Reels:      {date_range.get(\'count\')}")\n\n\ndef reels_to_dataframe(reels: list[dict]) -> pd.DataFrame:\n    """Build a DataFrame from reel metric dicts."""\n    rows = []\n    for r in reels:\n        rows.append({\n            "shortcode": r.get("shortcode"),\n            "owner": r.get("owner"),\n            "date": r.get("date"),\n            "views": r.get("views"),\n            "likes": r.get("likes"),\n            "comments": r.get("comments"),\n            "shares": r.get("shares"),\n            "saves": r.get("saves"),\n            "reposts": r.get("reposts"),\n            "status": r.get("status"),\n            "url": r.get("url") or r.get("reel_url"),\n        })\n    return pd.DataFrame(rows)\n\n\ndef show_reels_table(reels: list[dict], title: str = "Reel Metrics") -> pd.DataFrame:\n    """Print a formatted table of reel metrics and return the DataFrame."""\n    df = reels_to_dataframe(reels)\n    print_section(title)\n    if df.empty:\n        print("  No reels to display.")\n        return df\n    with pd.option_context(\n        "display.max_rows", None,\n        "display.max_columns", None,\n        "display.width", 120,\n        "display.max_colwidth", 40,\n    ):\n        print(df.to_string(index=True))\n    return df\n\n\ndef show_single_reel_metrics(metrics: dict) -> None:\n    """Print metrics for one reel as a key-value table."""\n    print_section("Single Reel Metrics")\n    fields = [\n        ("Shortcode", metrics.get("shortcode")),\n        ("Owner", metrics.get("owner")),\n        ("Date", metrics.get("date")),\n        ("Views", _fmt_int(metrics.get("views"))),\n        ("IG views", _fmt_int(metrics.get("ig_views"))),\n        ("Likes", _fmt_int(metrics.get("likes"))),\n        ("Comments", _fmt_int(metrics.get("comments"))),\n        ("Shares", _fmt_int(metrics.get("shares"))),\n        ("Saves", _fmt_int(metrics.get("saves"))),\n        ("Reposts", _fmt_int(metrics.get("reposts"))),\n        ("Video", metrics.get("is_video")),\n    ]\n    for label, value in fields:\n        print(f"  {label:12} {value}")\n    caption = (metrics.get("caption") or "").strip()\n    if caption:\n        preview = caption[:300] + ("…" if len(caption) > 300 else "")\n        print(f"  Caption:      {preview}")\n\n\ndef comments_to_dataframe(comments: list[dict]) -> pd.DataFrame:\n    rows = []\n    for c in comments:\n        rows.append({\n            "username": c.get("username"),\n            "text": c.get("text"),\n            "likes": c.get("likes"),\n            "date": c.get("date"),\n            "is_reply": c.get("is_reply"),\n            "parent_pk": c.get("parent_pk"),\n        })\n    return pd.DataFrame(rows)\n\n\ndef show_comments_table(\n    comments: list[dict],\n    title: str = "Comments",\n    max_rows: int = 50,\n) -> pd.DataFrame:\n    """Print comments (truncated) and return the full DataFrame."""\n    df = comments_to_dataframe(comments)\n    print_section(title)\n    print(f"  Total comments fetched: {len(df)}")\n    if df.empty:\n        print("  No comments.")\n        return df\n    display_df = df.head(max_rows)\n    with pd.option_context(\n        "display.max_rows", max_rows,\n        "display.max_columns", None,\n        "display.width", 120,\n        "display.max_colwidth", 60,\n    ):\n        print(display_df.to_string(index=True))\n    if len(df) > max_rows:\n        print(f"  … showing first {max_rows} of {len(df)} comments")\n    return df\n\n\ndef show_bulk_summary(summary: dict) -> None:\n    print_section("Bulk Fetch Summary")\n    print(f"  Total:      {summary.get(\'total\')}")\n    print(f"  Successful: {summary.get(\'successful\')}")\n    print(f"  Failed:     {summary.get(\'failed\')}")\n\n\ndef plot_reel_metrics(\n    reels: list[dict],\n    metrics: tuple[str, ...] = ("views", "likes", "comments"),\n    title: str = "Reel Metrics Comparison",\n) -> None:\n    """\n    Bar chart of numeric metrics per reel (matplotlib).\n\n    Works in Google Colab and local terminals with a display backend.\n    """\n    try:\n        import matplotlib.pyplot as plt\n    except ImportError:\n        print("matplotlib not installed — skipping chart.")\n        return\n\n    df = reels_to_dataframe(reels)\n    if df.empty:\n        return\n\n    labels = df["shortcode"].fillna(df.index.astype(str)).tolist()\n    x = range(len(labels))\n    width = 0.25\n\n    fig, ax = plt.subplots(figsize=(max(8, len(labels) * 0.6), 5))\n    offsets = [-width, 0, width]\n    plotted = 0\n    for i, metric in enumerate(metrics):\n        if metric not in df.columns:\n            continue\n        values = pd.to_numeric(df[metric], errors="coerce").fillna(0)\n        ax.bar(\n            [xi + offsets[plotted] for xi in x],\n            values,\n            width,\n            label=metric,\n        )\n        plotted += 1\n\n    if plotted == 0:\n        plt.close(fig)\n        return\n\n    ax.set_xticks(list(x))\n    ax.set_xticklabels(labels, rotation=45, ha="right")\n    ax.set_ylabel("Count")\n    ax.set_title(title)\n    ax.legend()\n    fig.tight_layout()\n    plt.show()\n', 'reel_metrics/api.py': '"""High-level API for fetching Instagram reel metrics."""\n\nfrom typing import Callable, Optional\n\nfrom instagrapi import Client\n\nfrom .errors import error_message_from_exception\nfrom .parsers import extract_shortcode, is_valid_media_input, resolve_target_username\nfrom .scraping import (\n    compute_reel_date_range,\n    fetch_media_comments,\n    fetch_single_media,\n    iter_user_clips_v1,\n    media_metric_fields,\n    media_to_dict,\n    media_to_reel_metrics,\n    reel_dict_without_comments,\n)\n\n\ndef process_reel_url(\n    cl: Client,\n    raw_code: str,\n    *,\n    fetch_comments: bool = True,\n) -> dict:\n    """Fetch metrics for one reel URL. Always returns a result dict with status."""\n    raw_code = raw_code.strip()\n    original_url = raw_code\n    base = {\n        "url": original_url,\n        "reel_url": original_url,\n        "views": None,\n        "likes": None,\n        "comments": None,\n        "shares": None,\n        "saves": None,\n        "reposts": None,\n        "date": None,\n        "status": "Failed",\n        "error": None,\n        "shortcode": None,\n        "caption": None,\n        "reel_comments": [],\n        "comments_fetched": 0,\n        "comments_note": None,\n    }\n    if not raw_code:\n        base["error"] = "Empty URL."\n        return base\n    if not is_valid_media_input(raw_code):\n        base["error"] = "Not a valid Instagram reel/post URL or shortcode."\n        return base\n\n    try:\n        shortcode = extract_shortcode(raw_code)\n        media_pk = cl.media_pk_from_code(shortcode)\n        m = fetch_single_media(cl, media_pk)\n        result = {\n            "url": original_url,\n            "reel_url": original_url,\n            "shortcode": m.get("shortcode") or shortcode,\n            "caption": m.get("caption") or "",\n            **media_metric_fields(m),\n            "reel_comments": [],\n            "comments_fetched": 0,\n            "comments_note": None,\n            "status": "Success",\n            "error": None,\n        }\n        if fetch_comments:\n            comments, comments_note = fetch_media_comments(cl, media_pk)\n            result["reel_comments"] = comments\n            result["comments_fetched"] = len(comments)\n            result["comments_note"] = comments_note\n        return result\n    except Exception as e:\n        base["reel_comments"] = []\n        base["comments_fetched"] = 0\n        base["comments_note"] = None\n        base["error"] = error_message_from_exception(e)\n        return base\n\n\ndef fetch_single_reel(\n    cl: Client,\n    reel_input: str,\n    *,\n    fetch_comments: bool = True,\n) -> dict:\n    """\n    Fetch full metrics for a single reel.\n\n    Returns a dict with metrics, comments (optional), and status fields.\n    Raises on login/network errors before scraping starts.\n    """\n    result = process_reel_url(cl, reel_input, fetch_comments=fetch_comments)\n    if result["status"] != "Success":\n        raise RuntimeError(result.get("error") or "Failed to fetch reel.")\n    return result\n\n\ndef fetch_profile_reels(\n    cl: Client,\n    target_raw: str,\n    limit: int = 20,\n    *,\n    on_reel: Optional[Callable[[int, dict], None]] = None,\n) -> dict:\n    """\n    List reels for a profile with per-reel metrics.\n\n    ``on_reel`` is called with (index, reel_dict) as each reel is fetched\n    (useful for progressive display in notebooks).\n    """\n    target = resolve_target_username(cl, target_raw)\n    if not target:\n        raise ValueError(\n            "Could not parse a username from the target. "\n            "Enter a plain username or profile URL."\n        )\n\n    user = cl.user_info_by_username_v1(target)\n    reels: list[dict] = []\n    reels_raw: list = []\n\n    if on_reel is not None:\n        for i, media in enumerate(iter_user_clips_v1(cl, user.pk, limit)):\n            reels_raw.append(media)\n            reel = media_to_reel_metrics(media)\n            reels.append(reel)\n            on_reel(i, reel)\n    else:\n        if limit > 0:\n            medias = cl.user_clips_v1(user.pk, amount=limit)\n        else:\n            medias = list(iter_user_clips_v1(cl, user.pk, limit=0))\n        reels_raw = medias\n        for media in medias:\n            try:\n                reels.append(reel_dict_without_comments(media_to_dict(media)))\n            except Exception as inner:\n                reels.append({\n                    "shortcode": getattr(media, "code", "?"),\n                    "date": None,\n                    "views": None,\n                    "likes": None,\n                    "comments": None,\n                    "caption": f"[error reading post: {inner}]",\n                    "is_video": None,\n                    "reel_comments": [],\n                    "comments_fetched": 0,\n                    "comments_note": "partial",\n                })\n\n    profile = {\n        "username": user.username,\n        "full_name": user.full_name,\n        "followers": user.follower_count,\n        "following": user.following_count,\n        "posts": user.media_count,\n        "is_private": user.is_private,\n    }\n\n    return {\n        "resolved_target": target,\n        "profile": profile,\n        "date_range": compute_reel_date_range(reels_raw),\n        "reels": reels,\n    }\n\n\ndef bulk_fetch_reels(\n    cl: Client,\n    urls: list[str],\n    *,\n    fetch_comments: bool = False,\n    on_progress: Optional[Callable[[int, int, dict], None]] = None,\n) -> dict:\n    """\n    Process multiple reel URLs.\n\n    ``on_progress`` receives (current, total, row_dict) after each URL.\n    """\n    results = []\n    successful = 0\n    failed = 0\n    total = len(urls)\n\n    for i, raw_url in enumerate(urls, start=1):\n        row = process_reel_url(cl, raw_url, fetch_comments=fetch_comments)\n        results.append(row)\n        if row["status"] == "Success":\n            successful += 1\n        else:\n            failed += 1\n        if on_progress:\n            on_progress(i, total, row)\n\n    return {\n        "results": results,\n        "summary": {\n            "total": len(results),\n            "successful": successful,\n            "failed": failed,\n        },\n    }\n', 'reel_metrics/__init__.py': '"""\nStandalone Instagram Reel Metrics — no web UI.\n\nFetch real Instagram Reel statistics (views, likes, comments, shares,\nsaves, reposts) using instagrapi\'s mobile API. Designed for scripts,\nterminals, and Google Colab.\n"""\n\nfrom .api import bulk_fetch_reels, fetch_profile_reels, fetch_single_reel, process_reel_url\nfrom .csv_io import parse_urls_from_csv, parse_urls_from_csv_path\nfrom .display import (\n    plot_reel_metrics,\n    show_bulk_summary,\n    show_comments_table,\n    show_profile_summary,\n    show_reels_table,\n    show_single_reel_metrics,\n)\nfrom .session import get_client, new_client, session_path\nfrom .scraping import fetch_media_comments, fetch_single_media\n\n__all__ = [\n    "bulk_fetch_reels",\n    "fetch_media_comments",\n    "fetch_profile_reels",\n    "fetch_single_media",\n    "fetch_single_reel",\n    "get_client",\n    "new_client",\n    "parse_urls_from_csv",\n    "parse_urls_from_csv_path",\n    "plot_reel_metrics",\n    "process_reel_url",\n    "session_path",\n    "show_bulk_summary",\n    "show_comments_table",\n    "show_profile_summary",\n    "show_reels_table",\n    "show_single_reel_metrics",\n]\n'}

def bootstrap_package():
    if os.path.isdir('reel_metrics') and os.path.isfile('reel_metrics/__init__.py'):
        return
    for rel_path, content in MODULES.items():
        os.makedirs(os.path.dirname(rel_path), exist_ok=True)
        with open(rel_path, 'w', encoding='utf-8') as f:
            f.write(content)
    print(f'Created {len(MODULES)} module files in reel_metrics/')

bootstrap_package()

if '.' not in sys.path:
    sys.path.insert(0, '.')
elif sys.path[0] != '.':
    sys.path.insert(0, '.')

import reel_metrics
importlib.reload(reel_metrics)
print('reel_metrics ready.')

In [ ]:
from reel_metrics import (
    bulk_fetch_reels,
    fetch_profile_reels,
    fetch_single_reel,
    get_client,
    parse_urls_from_csv,
    plot_reel_metrics,
    show_bulk_summary,
    show_comments_table,
    show_profile_summary,
    show_reels_table,
    show_single_reel_metrics,
)

print('All functions imported successfully.')

## Step 2 — Login

Set your Instagram credentials below. The password is only required on the first login in each Colab session (or when the session expires).

In [ ]:
from getpass import getpass

# ── CONFIGURE HERE ──────────────────────────────────────────────
IG_USERNAME = "your_instagram_username"  # login account (no @)

# Option A: type password here (less secure if you share the notebook)
IG_PASSWORD = ""

# Option B: leave IG_PASSWORD empty and uncomment the next line
# IG_PASSWORD = getpass("Instagram password: ")
# ─────────────────────────────────────────────────────────────────

try:
    client = get_client(IG_USERNAME, IG_PASSWORD or None)
    print(f"Authenticated as @{IG_USERNAME}")
except Exception as e:
    print(f"Login failed: {e}")
    print("Tip: set IG_PASSWORD and re-run this cell.")

---
## Mode 1 — Single Reel

Full metrics for one reel: views, likes, comments, shares, saves, reposts, plus comments list.

Accepted input: reel URL, post URL, or bare shortcode.

In [ ]:
# ── CONFIGURE HERE ──────────────────────────────────────────────
REEL_URL = "https://www.instagram.com/reel/EXAMPLE_SHORTCODE/"
FETCH_COMMENTS = True  # set False to skip comments (faster)
# ─────────────────────────────────────────────────────────────────

try:
    reel = fetch_single_reel(client, REEL_URL, fetch_comments=FETCH_COMMENTS)
    show_single_reel_metrics(reel)
    if FETCH_COMMENTS and reel.get('reel_comments'):
        df_comments = show_comments_table(reel['reel_comments'])
    else:
        print(f"\nComments: {reel.get('comments_fetched', 0)} fetched")
except Exception as e:
    print(f"Error: {e}")

---
## Mode 2 — Profile Reels

Fetch reels from a public Instagram profile.

| `LIMIT` | Behavior |
|---------|----------|
| `20` | First 20 reels (default) |
| `0` | All reels (slow for large accounts) |

In [ ]:
# ── CONFIGURE HERE ──────────────────────────────────────────────
TARGET = "atiazuhair"  # username, @username, profile URL, or reel URL
LIMIT = 20             # 0 = fetch all reels
SHOW_CHART = True
# ─────────────────────────────────────────────────────────────────

try:
    profile_data = fetch_profile_reels(
        client,
        TARGET,
        limit=LIMIT,
        on_reel=lambda i, r: print(f"  [{i + 1}] {r.get('shortcode')} — "
                                     f"views={r.get('views')} likes={r.get('likes')}"),
    )

    show_profile_summary(profile_data['profile'], profile_data['date_range'])
    df_profile = show_reels_table(
        profile_data['reels'],
        title=f"Profile Reels — @{profile_data['resolved_target']}",
    )

    if SHOW_CHART and profile_data['reels']:
        plot_reel_metrics(
            profile_data['reels'],
            title=f"@{profile_data['resolved_target']} — Reel Metrics",
        )
except Exception as e:
    print(f"Error: {e}")

---
## Mode 3 — Bulk CSV

Upload a CSV with reel URLs (one per row, or a column named `url` / `link` / `reel_url`).

**Example CSV:**
```
url
https://www.instagram.com/reel/ABC123/
https://www.instagram.com/reel/XYZ789/
```

In [ ]:
from google.colab import files

print('Select your CSV file...')
uploaded = files.upload()

if not uploaded:
    print('No file uploaded.')
else:
    csv_name = next(iter(uploaded))
    urls = parse_urls_from_csv(uploaded[csv_name])
    print(f'File: {csv_name}')
    print(f'URLs found: {len(urls)}')
    for i, u in enumerate(urls[:5], 1):
        print(f'  {i}. {u}')
    if len(urls) > 5:
        print(f'  ... and {len(urls) - 5} more')

In [ ]:
# Run bulk fetch (requires URLs from the upload cell above)
FETCH_BULK_COMMENTS = False  # True = much slower

try:
    _url_list = urls
except NameError:
    _url_list = []

if not _url_list:
    print('Upload a CSV first (run the cell above).')
else:
    try:
        bulk = bulk_fetch_reels(
            client,
            _url_list,
            fetch_comments=FETCH_BULK_COMMENTS,
            on_progress=lambda c, t, row: print(
                f"[{c}/{t}] {row.get('status')}: "
                f"{row.get('shortcode') or row.get('url')}"
                + (f" — {row.get('error')}" if row.get('error') else '')
            ),
        )

        show_bulk_summary(bulk['summary'])
        df_bulk = show_reels_table(bulk['results'], title='Bulk Fetch Results')

        successful = [r for r in bulk['results'] if r.get('status') == 'Success']
        if successful:
            plot_reel_metrics(successful, title='Bulk Fetch — Successful Reels')
    except Exception as e:
        print(f'Error: {e}')

---
## Export Results

Download the last profile or bulk results table as CSV.

In [ ]:
from google.colab import files
import os

export_path = 'instagram_reels_export.csv'

try:
    _df_bulk = df_bulk
except NameError:
    _df_bulk = None
try:
    _df_profile = df_profile
except NameError:
    _df_profile = None

if _df_bulk is not None and not _df_bulk.empty:
    _df_bulk.to_csv(export_path, index=False)
    print(f'Saved bulk results → {export_path} ({len(_df_bulk)} rows)')
elif _df_profile is not None and not _df_profile.empty:
    _df_profile.to_csv(export_path, index=False)
    print(f'Saved profile reels → {export_path} ({len(_df_profile)} rows)')
else:
    print('Run Profile Reels or Bulk CSV mode first to generate data.')
    export_path = None

if export_path and os.path.isfile(export_path):
    files.download(export_path)

---
## Troubleshooting

| Problem | Fix |
|---------|-----|
| `Login failed` | Set `IG_PASSWORD` and re-run login cell |
| `rate-limited` | Wait 10–15 min, reduce `LIMIT` or bulk size |
| `ChallengeRequired` | Open Instagram app, confirm login, retry |
| `Profile is private` | Log in with an account that follows that profile |
| `ModuleNotFoundError` | Re-run Cell 2 (bootstrap) |
| Runtime disconnected | Re-run from Cell 1 (install + bootstrap + login) |